# Sparkの遅延評価を体感で理解する

このノートブックでは、Spark UIを確認しながら遅延評価を体感します。

## 準備
1. クラスターにアタッチする
2. Spark UIを開く準備をする（クラスター名横のリンク、または右上の「View Spark UI」）

## 学習ポイント
- Transformation は「設計図」を作るだけ
- Action で初めて実行される
- Action のたびに再計算される
- cache() で再計算を防ぐ
- explain() で実行計画を見る

## 1. 遅延評価の確認：Transformationだけでは何も起きない

以下のセルを実行してください。巨大なデータに対して複数の処理を書いていますが、**一瞬で終わります**。

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

# このセルを実行しても一瞬で終わる
sdf = spark.read.format("delta").load("/databricks-datasets/nyctaxi/tables/nyctaxi_yellow")
sdf = sdf.filter(col("fare_amount") > 50)
sdf = sdf.withColumn("tip_rate", col("tip_amount") / col("fare_amount"))
sdf = sdf.groupBy("payment_type").agg(
    F.count("*").alias("count"),
    F.avg("tip_rate").alias("avg_tip_rate")
)

print("✅ Transformation完了（でも何も実行されていない）")
print("")
print("👉 Spark UI > Jobs タブを確認してください")
print("   → ジョブが増えていないはず！")

✅ Transformation完了（でも何も実行されていない）

👉 Spark UI > Jobs タブを確認してください
   → ジョブが増えていないはず！


### 🔍 確認ポイント

1. Spark UI を開く
2. **Jobs** タブを確認
3. ジョブが増えていないことを確認

これが「遅延評価」です。Transformation は設計図を作るだけで、実際の処理は行われていません。

## 2. Actionで初めて実行される

`show()` を実行すると、ここで初めてジョブが走ります。

In [0]:
# ここで初めてジョブが走る
print("=== show() を実行 ===")
sdf.show()

print("")
print("👉 Spark UI > Jobs タブを確認してください")
print("   → ジョブが1つ増えたはず！")

=== show() を実行 ===
+------------+-------+--------------------+
|payment_type|  count|        avg_tip_rate|
+------------+-------+--------------------+
|         NA |    294| 0.00101149357833006|
|         CSH|3835235|2.169227226815231...|
|   No Charge|   3441|0.003350421579143...|
|         DIS|  18844|0.001616262978891...|
|         CRD|6589771| 0.17908416374872954|
|         Cre| 113897| 0.14130016384219107|
|        CASH| 114710|5.414680606942378E-7|
|         CAS|  50567|9.078644323087296E-5|
|         Dis|    398|0.002811996966359...|
|         UNK|  16932| 0.17539690013471898|
|         CRE|  14408| 0.14617670324808457|
|      Credit| 202146| 0.13762811084025553|
|         NOC|  55129|0.001777176394998...|
|         Cas|  34813|9.547683717789813E-4|
|        Cash|  74534|0.001054568332993706|
|     Dispute|   1004|0.001947189241313...|
|         No |   1208|0.003141984155836...|
|      CREDIT|  12489| 0.13645576016405664|
|           3|  44708|3.926530149692506...|
|           1

### 🔍 確認ポイント

1. Spark UI の **Jobs** タブにジョブが表示されている
2. Stages タブで複数のステージが確認できる
3. 「さっきの全処理（read → filter → withColumn → groupBy → agg）がここで一気に実行された」ことを実感

## 3. Actionのたびに再計算される（重要！）

ここが最も重要なポイントです。**Actionを呼ぶたびに、最初から再計算されます**。

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F
import time

# データ準備（Transformation）
sdf = spark.read.format("delta").load("/databricks-datasets/nyctaxi/tables/nyctaxi_yellow")
sdf = sdf.filter(col("fare_amount") > 50)
sdf = sdf.withColumn("tip_rate", col("tip_amount") / col("fare_amount"))
sdf = sdf.groupBy("payment_type").agg(
    F.count("*").alias("count"),
    F.avg("tip_rate").alias("avg_tip_rate")
)

print("=" * 50)
print("=== 1回目: show() ===")
print("=" * 50)
start = time.time()
sdf.show()
time1 = time.time() - start
print(f"⏱️ 所要時間: {time1:.2f}秒\n")

print("=" * 50)
print("=== 2回目: count() ===")
print("=" * 50)
start = time.time()
cnt = sdf.count()
print(f"件数: {cnt}")
time2 = time.time() - start
print(f"⏱️ 所要時間: {time2:.2f}秒\n")

print("=" * 50)
print("=== 3回目: collect() ===")
print("=" * 50)
start = time.time()
data = sdf.collect()
print(f"取得件数: {len(data)}")
time3 = time.time() - start
print(f"⏱️ 所要時間: {time3:.2f}秒\n")

print("=" * 50)
print("👉 Spark UI > Jobs タブを確認してください")
print("   → ジョブが3つ増えたはず（毎回最初から再計算！）")
print("=" * 50)

=== 1回目: show() ===
+------------+-------+--------------------+
|payment_type|  count|        avg_tip_rate|
+------------+-------+--------------------+
|         NA |    294| 0.00101149357833006|
|         CSH|3835235|2.169227226815231...|
|   No Charge|   3441|0.003350421579143...|
|         DIS|  18844|0.001616262978891...|
|         CRD|6589771| 0.17908416374872954|
|         Cre| 113897| 0.14130016384219107|
|        CASH| 114710|5.414680606942378E-7|
|         CAS|  50567|9.078644323087296E-5|
|         Dis|    398|0.002811996966359...|
|         UNK|  16932| 0.17539690013471898|
|         CRE|  14408| 0.14617670324808457|
|      Credit| 202146| 0.13762811084025553|
|         NOC|  55129|0.001777176394998...|
|         Cas|  34813|9.547683717789813E-4|
|        Cash|  74534|0.001054568332993706|
|     Dispute|   1004|0.001947189241313...|
|         No |   1208|0.003141984155836...|
|      CREDIT|  12489| 0.13645576016405664|
|           3|  44708|3.926530149692506...|
|           

### 🔍 確認ポイント

- Spark UI の **Jobs** タブにジョブが **3つ** 追加されている
- それぞれ似たような処理時間がかかっている
- 変数 `sdf` に入れても、結果は保持されていない（設計図への参照でしかない）

**これが pandas と Spark の大きな違いです！**

> **補足: Spark UIで「skipped」と表示される理由**
> 
> Spark UIでタスクが「skipped」と表示されることがあります。これは主に以下の理由によるものです：
> 
> - **Delta Lakeのデータスキッピング**: Deltaはファイルごとに統計情報（min/max値など）を保持しており、フィルタ条件を満たすデータがないファイルは読み込み自体をスキップします
> - **Adaptive Query Execution (AQE)**: Spark 3.0以降でデフォルト有効な機能で、実行時にクエリプランを動的に最適化し、空のパーティションなど不要なタスクをスキップします
> - **キャッシュヒット**: `cache()`済みのデータを読む場合、元データの読み込みステージがスキップされます
> 
> これらの最適化により、見た目上のジョブ数やタスク数は変動しますが、「Actionのたびに再計算が発生する」という本質は変わりません。

## 4. cache()で再計算を防ぐ

`cache()` を使うと、最初のAction時に結果をメモリに保持し、以降のActionではキャッシュから読み込みます。

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F
import time

# データ準備（Transformation）
sdf = spark.read.format("delta").load("/databricks-datasets/nyctaxi/tables/nyctaxi_yellow")
sdf = sdf.filter(col("fare_amount") > 50)
sdf = sdf.withColumn("tip_rate", col("tip_amount") / col("fare_amount"))
sdf = sdf.groupBy("payment_type").agg(
    F.count("*").alias("count"),
    F.avg("tip_rate").alias("avg_tip_rate")
)

# ★ キャッシュを指定（これ自体はTransformationなので何も起きない）
sdf.cache()

print("=" * 50)
print("=== 1回目: show() (計算してキャッシュに保存) ===")
print("=" * 50)
start = time.time()
sdf.show()
time1 = time.time() - start
print(f"⏱️ 所要時間: {time1:.2f}秒\n")

print("=" * 50)
print("=== 2回目: count() (キャッシュから読む) ===")
print("=" * 50)
start = time.time()
cnt = sdf.count()
print(f"件数: {cnt}")
time2 = time.time() - start
print(f"⏱️ 所要時間: {time2:.2f}秒\n")

print("=" * 50)
print("=== 3回目: collect() (キャッシュから読む) ===")
print("=" * 50)
start = time.time()
data = sdf.collect()
print(f"取得件数: {len(data)}")
time3 = time.time() - start
print(f"⏱️ 所要時間: {time3:.2f}秒\n")

print("=" * 50)
print(f"📊 比較: 1回目 {time1:.2f}秒 → 2回目 {time2:.2f}秒 → 3回目 {time3:.2f}秒")
print("👉 2回目以降が明らかに速い！")
print("")
print("👉 Spark UI > Storage タブを確認してください")
print("   → RDDs セクションにキャッシュが表示されている")
print("=" * 50)

=== 1回目: show() (計算してキャッシュに保存) ===
+------------+-------+--------------------+
|payment_type|  count|        avg_tip_rate|
+------------+-------+--------------------+
|      -52.19|      1|                 0.0|
|         NA |    294| 0.00101149357833006|
|      -54.58|      1|                 0.0|
|           3|  44708|3.926530149692506...|
|      -58.64|      1|                 0.0|
|         CSH|3835235|2.169227226815231...|
|   No Charge|   3441|0.003350421579143...|
|           0|     42|                 0.0|
|         DIS|  18844|0.001616262978891...|
|      -50.56|      1|                 0.0|
|         -58|      1|                 0.0|
|      -53.71|      1|                 0.0|
|           5|      3|                 0.0|
|         CRD|6589771| 0.17908416374872954|
|         Cre| 113897| 0.14130016384219107|
|        CASH| 114710|5.414680606942378E-7|
|         CAS|  50567|9.078644323087296E-5|
|      -50.64|      1|                 0.0|
|       -58.5|      1|                 0.

In [0]:
# キャッシュを解放
sdf.unpersist()
print("✅ キャッシュを解放しました")

✅ キャッシュを解放しました


### 🔍 確認ポイント

1. **処理時間を比較**: 2回目以降が明らかに速い
2. **Storage タブ**: **RDDs** セクションにキャッシュされたDataFrameが表示される（Storage Level: Disk Memory Deserialized 1x Replicated）
3. **Jobs タブ**: 2回目以降のジョブでは、多くのStageがスキップされている

## 5. explain()で実行計画を見る

`explain()` を使うと、**Actionを実行せずに**実行計画（設計図）だけを見ることができます。

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

sdf = spark.read.format("delta").load("/databricks-datasets/nyctaxi/tables/nyctaxi_yellow")
sdf = sdf.filter(col("fare_amount") > 50)
sdf = sdf.withColumn("tip_rate", col("tip_amount") / col("fare_amount"))
sdf = sdf.groupBy("payment_type").agg(F.avg("tip_rate").alias("avg_tip_rate"))

print("=" * 60)
print("=== 基本の実行計画 ===")
print("（下から上に読む）")
print("=" * 60)
sdf.explain()

=== 基本の実行計画 ===
（下から上に読む）
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonGroupingAgg(keys=[payment_type#824], functions=[finalmerge_avg(merge sum#904, count#905L) AS avg(tip_rate#850)#890])
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage
               +- PhotonShuffleExchangeSink hashpartitioning(payment_type#824, 200)
                  +- PhotonGroupingAgg(keys=[payment_type#824], functions=[partial_avg(tip_rate#850) AS (sum#904, count#905L)])
                     +- PhotonProject [payment_type#824, (tip_amount#828 / fare_amount#825) AS tip_rate#850]
                        +- PhotonScan parquet [payment_type#824,fare_amount#825,tip_amount#828] DataFilters: [isnotnull(fare_amount#825), (fare_amount#825 > 50.0)], DictionaryFilters: [(fare_amount#825 > 50.0)], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yello

In [0]:
print("=" * 60)
print("=== フォーマット済み実行計画 ===")
print("=" * 60)
sdf.explain("formatted")

=== フォーマット済み実行計画 ===
== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   ColumnarToRow (9)
   +- PhotonResultStage (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [3]: [payment_type#824, fare_amount#825, tip_amount#828]
DictionaryFilters: [(fare_amount#825 > 50.0)]
Location: PreparedDeltaFileIndex [dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow]
ReadSchema: struct<payment_type:string,fare_amount:double,tip_amount:double>
RequiredDataFilters: [isnotnull(fare_amount#825), (fare_amount#825 > 50.0)]

(2) PhotonProject
Input [3]: [payment_type#824, fare_amount#825, tip_amount#828]
Arguments: [payment_type#824, (tip_amount#828 / fare_amount#825) AS tip_rate#850]

(3) PhotonGroupingAg

### 読み方のポイント

- **下から上に読む**（データの流れ）
- Photon有効環境では各オペレータに`Photon`プレフィックスが付く
- `PhotonScan` → `PhotonProject` → `PhotonGroupingAgg`
- `DataFilters`はPredicate Pushdownが効いている証拠（FilterがScanに組み込まれている）
- `PhotonShuffleExchange`はシャッフル（ネットワーク転送）を意味する

## 6. よくある誤解

| ❌ 誤解 | ✅ 正しい理解 |
|--------|-------------|
| 変数に入れたら計算される | 変数はただの設計図への参照 |
| filterしたらデータが減る | Actionまで何も起きない |
| 2回showしても同じ結果だから1回分 | 2回計算している |
| cache()したら即座にメモリに載る | 最初のActionで初めてキャッシュされる |

## 7. 実践Tips

### 開発時のTips: サンプルで開発、本番で全データ

In [0]:
from pyspark.sql.functions import col

# 全データを読み込むが...
sdf_full = spark.read.format("delta").load("/databricks-datasets/nyctaxi/tables/nyctaxi_yellow")

# サンプルを作ってキャッシュ（開発用）
sdf_sample = sdf_full.limit(10000).cache()
print(f"サンプルサイズ: {sdf_sample.count():,} 件")

# 開発中はサンプルで高速に試行錯誤
sdf_result = sdf_sample.filter(col("fare_amount") > 50)
sdf_result.show(5)

# サンプルのキャッシュを解放
sdf_sample.unpersist()

サンプルサイズ: 10,000 件
+---------+-------------------+-------------------+---------------+-------------+----------------+---------------+------------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|rate_code_id|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|total_amount|
+---------+-------------------+-------------------+---------------+-------------+----------------+---------------+------------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+------------+
|      VTS|2009-11-30 14:53:00|2009-11-30 14:55:00|              1|         0.01|      -76.369953|       38.75836|        NULL|              NULL|       -76.369898|       38.758257|      Credit|       55.0|  

DataFrame[vendor_id: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, passenger_count: int, trip_distance: double, pickup_longitude: double, pickup_latitude: double, rate_code_id: int, store_and_fwd_flag: string, dropoff_longitude: double, dropoff_latitude: double, payment_type: string, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, total_amount: double]

### 本番前のチェック: explain()でプランを確認

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

sdf = spark.read.format("delta").load("/databricks-datasets/nyctaxi/tables/nyctaxi_yellow")
sdf = sdf.filter(col("fare_amount") > 50)
sdf = sdf.groupBy("payment_type").agg(F.sum("fare_amount").alias("total"))

print("=== 実行前にプランを確認 ===")
print("確認ポイント:")
print("  - PhotonShuffleExchange（シャッフル）の数")
print("  - DataFilters にフィルタ条件が含まれているか")
print("  - ReadSchema の読み込むカラム数")
print("")
sdf.explain("formatted")

=== 実行前にプランを確認 ===
確認ポイント:
  - PhotonShuffleExchange（シャッフル）の数
  - DataFilters にフィルタ条件が含まれているか
  - ReadSchema の読み込むカラム数

== Physical Plan ==
AdaptiveSparkPlan (9)
+- == Initial Plan ==
   ColumnarToRow (8)
   +- PhotonResultStage (7)
      +- PhotonGroupingAgg (6)
         +- PhotonShuffleExchangeSource (5)
            +- PhotonShuffleMapStage (4)
               +- PhotonShuffleExchangeSink (3)
                  +- PhotonGroupingAgg (2)
                     +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [2]: [payment_type#2163, fare_amount#2164]
DictionaryFilters: [(fare_amount#2164 > 50.0)]
Location: PreparedDeltaFileIndex [dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow]
ReadSchema: struct<payment_type:string,fare_amount:double>
RequiredDataFilters: [isnotnull(fare_amount#2164), (fare_amount#2164 > 50.0)]

(2) PhotonGroupingAgg
Input [2]: [payment_type#2163, fare_amount#2164]
Arguments: [payment_type#2163], [partial_sum(fare_amount#2164) AS sum#2221], [sum#2220], 

## まとめ

| ポイント | 確認方法 |
|---------|---------|
| Transformationは何もしない | Spark UI > Jobs で確認 |
| Actionで実行される | Spark UI > Jobs で確認 |
| 毎回再計算される | 実行時間を比較 |
| cache()で高速化 | 実行時間を比較、Storage タブ |
| 実行計画の確認 | explain() |

---

**遅延評価を理解すれば、Sparkの動きが見えてくる！**